# Title: Regenerative Cooling Engine

Date: 24 Apr 2025



-------

## Summary and Problem Statement
In liquid-fuel rocket engines, a critical design consideration is the temperature of the combustion chamber walls, as the heat generated by fuel combustion can melt through the chamber walls if they are thermally unregulated. A common solution is regenerative cooling, where some fuel is pumped into the combustion chamber through holes in the chamber walls called cooling channels, absorbing heat from the combustion chamber walls en route to the combustion chamber. This simultaneously regulates the chamber wall temperature and makes the combustion process more efficient.

Our simple cycle aims to determine the thrust and whether or not the wall material melts under the combustion process.

-------

## Background information
We've attempted to model the thermodynamic process of liquid-fuel combustion and determine the temperature that the inner chamber reaches. We've included the melting points of different materials, to display their effectiveness. Three common fuel chamber materials are listed below:

Copper = 1358K, Stainless Steel = 1800K, T6060 Aluminum Alloy = 960K.

Because the cycle must start in the liquid phase, we are using the Rankine cycle to model the system

Another modification we've added, is the innercooler which is an isobaric process that reduces the heat, q_out. The intercooler acts between the two compressors. as a way to release heat.

Rankine cycle-based systems: 15-35% for liquid-fueled engines using coolant phase changes

#### Equations

Heat Exchanger
$$ Q_{out} = m c_p \Delta T $$

Isentropic Efficiency
$$ \eta_{isentropic} = \frac{w_{isentropic}}{w_{real}} = \frac{h_{2,isentropic} - h_1}{h_{2,actual} - h_1} $$

Isentropic Ideal Gas Relations
$$
\Delta s = c_p ln \frac{T_2}{T_1} + R_u ln \frac{v_2}{v_1} = c_v ln \frac{T_2}{T_1} + R_u ln \frac{P_2}{P_1}
$$ $$
(\frac{T_2}{T_1})_{s = const.} = (\frac{v_1}{v_2}) ^ {k-1}
$$ $$
(\frac{T_2}{T_1})_{s = const.} = (\frac{P_2}{P_1}) ^ {(k-1) / k}
$$ $$
(\frac{P_2}{P_1})_{s = const.} = (\frac{v_1}{v_2}) ^ {k}
$$

Regenerator Effectiveness
$$
\epsilon = \frac{q_{regen,act}}{q_{regen,max}} = \frac{h_5 - h_2}{h_4 - h_2}
$$

-------

## Modeling the Thermodynamic Process



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
!pip3 install sympy
import sympy as sp
!pip3 install CoolProp
from CoolProp.CoolProp import PropsSI
import CoolProp.CoolProp as CP

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 73.9 MB/s eta 0:00:00


# Assumption:
Each isentropic component has a range of isentropic efficiency. For this analysis, we will use the minimum isentropic efficiency for each component for consistency throughout the problem.

In [ ]:
import pandas as pd

data = {'Component': ['Compressor', 'Turbine', 'Nozzle'],
        'Isentropic Efficiency Range': ['0.75-0.90', '0.85-0.95', '0.90-0.98'],
        'Impact on Cycle': ['↑ Work Input', '↓ Work Output', '↓ Thrust']}

eta_pump = 0.75
eta_turb = 0.85
eta_nozzle = 0.9
df = pd.DataFrame(data)
df


,Component,Isentropic Efficiency Range,Impact on Cycle
0,Compressor,0.75-0.90,↑ Work Input
1,Turbine,0.85-0.95,↓ Work Output
2,Nozzle,0.90-0.98,↓ Thrust


# **Step 0:**
Set up ethanol initial conditions as ambient conditions at atmospheric pressure and 25 degrees celsius. We've defined the quality as 0, because ethanol is a working fluid, so it must always be in the liquid state. We've also defined the pressure under ambient atmospheric conditions.

In [ ]:
# Ambient Conditions
fluid = 'ethanol'
P_1 = 101325
P1 = 101325       # Initial pressure (Pa) - 1 atm
T_1 = PropsSI('T', 'P', 101325, 'Q', 0, fluid); print(f'temperature of eth = {T_1:.3f} J/kgK') # K
T1 = T_1

temperature of eth = 351.570 J/kgK


Other Assumptions:

The ethanol is pressurized before entering the chamber. In addition, the Cp of ethanol is found online.

In [ ]:
copper_max_temp = 1358
steel_max_temp = 1800
aluminum_max_temp = 960

#Cooling system
channel_length = 0.8  # m
channel_area = 0.0001  # m²
num_channels = 50
copper_thickness = 0.005  # m
P_ethanol = 2.5e6       # Pa (25 bar)
ethanol_mass_flow = 1.5  # kg/s
cp_eth = 2570 #J/kgK

# **Step 1:**
Find maximum pressure and temperature so that ethanol is in liquid phase

# **State 1:**

Initial Pressure and Temperature = Atmospheric Temperature/Pressure

Assumption is that the inlet velocity is so small that is has no effect on the enthalpy. Similarly, we assume the ethanol behaves as a working fluid, so Q=0 for all our equations. We find pressure, enthalpy, entropy, and specific volume(1/Density) based on the defined state of room temperature(300K) and the ethanol as a saturated fluid.

In [ ]:
T_cond=300
# State 1: Saturated liquid at condenser exit
P_low = CP.PropsSI('P', 'T', T_cond, 'Q', 0, fluid)
h1 = CP.PropsSI('H', 'T', T_cond, 'Q', 0, fluid)
s1 = CP.PropsSI('S', 'T', T_cond, 'Q', 0, fluid)
v1 = 1/CP.PropsSI('D', 'T', T_cond, 'Q', 0, fluid)


# **State 2:**

*   Use Atmospheric Temperature/Pressure to find the initial pressure and enthalpy.
*   Then, use isentropic pump efficiency to manipulate isentropic enthalpy and temperature at constant entropy to find actual temperatures and work




In [ ]:
P2 = 3 * P_1  # Pressure ratio 3:1
h2s = CP.PropsSI('H', 'P', P2, 'S', s1, 'ETHANOL')
T2s = CP.PropsSI('T', 'P', P2, 'S', s1, 'ETHANOL')
s1 = CP.PropsSI('S','T',T_1,'P',P_low,'ETHANOL')
# Calculate actual enthalpy at state 2
h2 = h1 + (h2s - h1) / eta_pump  # [J/kg]
# Estimate actual temperature using approximate Cp for liquid ethanol
Cp = 2570  # [J/kg·K] - specific heat capacity assumption
T_2_act = T_1 + (h2 - h1)/Cp  # [K]
P_2 = P2

print(f'T2 actual = {T_2_act:.3f} K')

T2 actual = 351.766 K



# **State 3:**

The combustion chamber, which takes in heat at an isobaric process. The temperature of state 3 slightly decreases due to the efficiency of the heat exchanger. We know that the heat that enters the system during this process is equal to mass flow rate times the change in enthalpy from first law of thermodynamics(work done is zero).

In [ ]:
P3=P2
T3 = 411.77  # K (80% of T_crit)
T_3a = (T_2_act - 0.05 * T3) / 0.95 #0.95 effectiveness
h3 = CP.PropsSI('H', 'T', T3, 'P', P2, 'ETHANOL')
Q_in = ethanol_mass_flow*(h3 - h2)
print(f'Qin = {Q_in:.3f} K')
T_boil_eth = PropsSI('T', 'Q', 1, 'P', P_2, fluid); print(f'boiling temp of eth = {T_boil_eth:.3f} K')

print("eth temp above boiling") if T_boil_eth <= T3 else print(f"below boiling by {T_boil_eth - T3:.3f}")

Qin = 1625065.567 K
boiling temp of eth = 382.284 K
eth temp above boiling



# **State 4:**

Isentropic turbine, the system closes at the initial pressure P1. We can find the entropy of state 3 based on the pressure of state 2(since 2-3 is isobaric). Similarly, since 3-4 is an isentropic process, we can use the entropy of state 3 and the pressure p1 to define the rest of the state variables at state 4.

In [ ]:

s3 = CP.PropsSI('S', 'T', T_3a, 'P', P2, fluid)
P4 = P1  # Cycle closes at initial pressure
T4 = CP.PropsSI('T', 'P', P4, 'S', s3, fluid)
print(T4)


348.56411167687867



# **State 4-1:**

Heat rejection, Q_out = Cp * (T3a-T3) assumes that ethanol behaves as an ideal gas in this scenario after it has been superheated. The enthalpy of state four is equivalent to the kinetic energy, which we can find the velocity of superheated ethanol from.

In [ ]:
import numpy as np
q_out = cp_eth * (T_3a - T3)
q_nozzle = eta_nozzle* q_out
h4 = CP.PropsSI('H', 'T|gas', T4, 'P', P4, fluid)
v_out_eth = np.sqrt(2 * h4); print(f'velocity = {v_out_eth:.3f} m/s')

velocity = 1472.000 m/s


In [ ]:
compressor_work = h2 - h1
turbine_work = h3 - h4
heat_added = h3 - h2
net_work = turbine_work - compressor_work
efficiency = net_work / heat_added
phases = {

        'State 1': CP.PhaseSI('T', T1, 'P', P1, fluid),
        'State 2': CP.PhaseSI('T', T_2_act, 'P', P2, fluid),
        'State 3': CP.PhaseSI('T', T3, 'P', P2, fluid),
        'State 4': CP.PhaseSI('T', T4, 'P', P4, fluid)
    }

print(efficiency, phases, net_work)

0.09294559654090855 {'State 1': 'unknown: Saturation pressure [101325 Pa] corresponding to T [351.57 K] is within 1e-4 % of given p [101325 Pa] : PropsSI("Phase","T",351.5704045,"P",101325,"ethanol")', 'State 2': 'liquid', 'State 3': 'gas', 'State 4': 'liquid'} 100695.12570213014



# **State 6-1:**

Heat rejection
Cool exhaust to the environment
Heat rejection: Q out = H6-H1

In [ ]:
T6 = CP.PropsSI('T', 'P', 101325, 'S', s3, fluid)
h_exit = CP.PropsSI('H', 'T', 803.4, 'P', 101325, fluid)  # From previous cycle
vexit = np.sqrt(2*(h_exit-h4))

thrust = ethanol_mass_flow*vexit
print(f'The thrust for the rocket is {thrust:.3f} Newtons.')

The thrust for the rocket is 2111.723 Newtons.


In [ ]:
Q_comb = CP.PropsSI('C','T',T3,'P',P2,'Air') * (T3 - T_2_act)
P_ethanol = 2.5e6  # Pa (25 bar)
channel_length = 0.8  # m
channel_area = 0.0001  # m²
num_channels = 50
def copper_wall_cooling(Q_combustion):
    # Thermal parameters
    surface_area = num_channels * channel_length * (4*channel_area/np.pi)**0.5
    q_flux = Q_combustion * 0.4 / surface_area  # 40% heat to walls

    # Ethanol properties (enforce liquid phase)
    T_eth_in = CP.PropsSI('T','P',P_ethanol,'Q',0,ethanol+'|liquid') - 15  # 15K subcool
    cp_eth = CP.PropsSI('C','P',P_ethanol,'T',T_eth_in,ethanol+'|liquid')

    # Temperature rise calculation
    delta_T_eth = (Q_combustion * 0.4) / (ethanol_mass_flow * cp_eth)
    T_eth_out = T_eth_in + delta_T_eth

    # Wall temperature gradient
    delta_T_wall = q_flux * wall_thickness / copper_k
    T_wall_hot = T_eth_out + delta_T_wall

    # Validation checks
    if T_wall_hot > max_wall_temp:
        raise ValueError(f"Copper wall overheated: {T_wall_hot:.1f}K > {max_wall_temp}K")

    T_sat = CP.PropsSI('T','P',P_ethanol,'Q',0,ethanol)
    if T_eth_out >= T_sat:
        raise ValueError(f"Ethanol boiling at {T_eth_out:.1f}K (Sat: {T_sat:.1f}K)")

    return T_wall_hot, delta_T_eth

    T_wall, eth_dT = copper_wall_cooling(Q_comb)

# Thermal efficiency calculation
work_comp = CP.PropsSI('H','P',101325*3,'S',s1,fluid) - CP.PropsSI('H','T',T1,'P',P_low,fluid)
work_turb = CP.PropsSI('H','T',T3,'P',101325*3,fluid) - CP.PropsSI('H','T',T4,'P',101325,fluid)
thermal_eff = (work_turb - work_comp) / (100*Q_comb)
print(f'Thermal efficiency is {thermal_eff:.3f} ')

Thermal efficiency is 0.112 


-----

## Improving the performance

Changing the working fluid from ethanol to methanol because methanol has a higher specific heat. Methanol absorbs more energy per kg when vaporizing, which means methanol removes more heat than ethanol at the same mass flow rate. The code below is the same as all the code above, with the working fluid changed.

In [ ]:
fluid = 'methanol'
P_1 = 101325
P1 = 101325      # Initial pressure (Pa) - 1 atm
T_1 = PropsSI('T', 'P', P1, 'Q', 0, fluid)
T1 = T_1

copper_max_temp = 1358
steel_max_temp = 1800
aluminum_max_temp = 960

#Cooling system
channel_length = 0.8  # m
channel_area = 0.0001  # m²
num_channels = 50
copper_thickness = 0.005  # m
P_ethanol = 2.5e6       # Pa (25 bar)
ethanol_mass_flow = 1.5  # kg/s
cp_eth = 2570

T_cond=300
# State 1: Saturated liquid at condenser exit
P_low = CP.PropsSI('P', 'T', T_cond, 'Q', 0, fluid)
h1 = CP.PropsSI('H', 'T', T_cond, 'Q', 0, fluid)
s1 = CP.PropsSI('S', 'T', T_cond, 'Q', 0, fluid)
v1 = 1/CP.PropsSI('D', 'T', T_cond, 'Q', 0, fluid)

P2 = 3 * P_1
h2s = CP.PropsSI('H', 'P', P2, 'S', s1, fluid)
T2s = CP.PropsSI('T', 'P', P2, 'S', s1, fluid)
s1 = CP.PropsSI('S','T',T_1,'P',P_low, fluid)
# Calculate actual enthalpy at state 2
h2 = h1 + (h2s - h1) / eta_pump  # [J/kg]
# Estimate actual temperature using approximate Cp for liquid ethanol
Cp = 2570  # [J/kg·K] - specific heat capacity assumption
T_2_act = T_1 + (h2 - h1)/Cp  # [K]
P_2 = P2

P3=P2
T3 = 411.57
T_3a = (T_2_act - 0.05 * T3) / 0.95 #0.95 effectiveness
h3 = CP.PropsSI('H', 'T', T3, 'P', P2, fluid)
Q_in = ethanol_mass_flow*(h3 - h2)

T_boil_eth = PropsSI('T', 'Q', 1, 'P', P_2, fluid)


s3 = CP.PropsSI('S', 'T', T_3a, 'P', P2, fluid)
P4 = P1  # Cycle closes at initial pressure
T4 = CP.PropsSI('T', 'P', P4, 'S', s3, fluid)

import numpy as np
q_out = cp_eth * (T_3a - T3)
q_nozzle = eta_nozzle* q_out
h4 = CP.PropsSI('H', 'T|gas', T4, 'P', P4, fluid)
v_out_eth = np.sqrt(2 * h4)

compressor_work = h2 - h1
turbine_work = h3 - h4
heat_added = h3 - h2
net_work = turbine_work - compressor_work
efficiency = net_work / heat_added
phases = {

        'State 1': CP.PhaseSI('T', T1, 'P', P1, fluid),
        'State 2': CP.PhaseSI('T', T_2_act, 'P', P2, fluid),
        'State 3': CP.PhaseSI('T', T3, 'P', P2, fluid),
        'State 4': CP.PhaseSI('T', T4, 'P', P4, fluid)
    }


T6 = CP.PropsSI('T', 'P', P1, 'S', s3, fluid)
h_exit = CP.PropsSI('H', 'T', 803.4, 'P', P1, fluid)  # From previous cycle
vexit = np.sqrt(2*(h_exit-h4)) #changed h5 to h4
new_thrust = ethanol_mass_flow*vexit

Q_comb = CP.PropsSI('C','T',T3,'P',P2,'Air') * (T3 - T_2_act)
P_methanol = 2.5e6  # Pa (25 bar)
channel_length = 0.8  # m
channel_area = 0.0001  # m²
num_channels = 50
def copper_wall_cooling(Q_combustion):
    # Thermal parameters
    surface_area = num_channels * channel_length * (4*channel_area/np.pi)**0.5
    q_flux = Q_combustion * 0.4 / surface_area  # 40% heat to walls

    # Ethanol properties (enforce liquid phase)
    T_eth_in = CP.PropsSI('T','P',P_methanol,'Q',0,fluid+'|liquid') - 15  # 15K subcool
    cp_eth = CP.PropsSI('C','P',P_methanol,'T',T_eth_in,fluid+'|liquid')

    # Temperature rise calculation
    delta_T_eth = (Q_combustion * 0.4) / (ethanol_mass_flow * cp_eth)
    T_eth_out = T_eth_in + delta_T_eth

    # Wall temperature gradient
    delta_T_wall = q_flux * wall_thickness / copper_k
    T_wall_hot = T_eth_out + delta_T_wall

    # Validation checks
    if T_wall_hot > max_wall_temp:
        raise ValueError(f"Copper wall overheated: {T_wall_hot:.1f}K > {max_wall_temp}K")

    T_sat = CP.PropsSI('T','P',P_ethanol,'Q',0,ethanol)
    if T_eth_out >= T_sat:
        raise ValueError(f"Methanol boiling at {T_eth_out:.1f}K (Sat: {T_sat:.1f}K)")

    return T_wall_hot, delta_T_eth

    T_wall, eth_dT = copper_wall_cooling(Q_comb)

# Thermal efficiency calculation
work_comp = CP.PropsSI('H','P',P1*3,'S',s1,fluid) - CP.PropsSI('H','T',T1,'P',P_low,fluid)
work_turb = CP.PropsSI('H','T',T3,'P',P1*3,fluid) - CP.PropsSI('H','T',T4,'P',P1,fluid)
new_thermal_eff = (work_turb - work_comp) / (100*Q_comb)
print(f'improved thermal efficiency for methanol is {new_thermal_eff:.3f} where the previous thermal efficiency with ethanol was {thermal_eff:.3f}')



improved thermal efficiency for methanol is 0.125 where the previous thermal efficiency with ethanol was 0.112


----

## Conclusions (150 words max)

The maximum wall temperature is 489.9K, which is significantly below the melting point for Copper(1358K). The maximum wall temperature is also below the melting point of steel and aluminum(960K). In consideration of the weight of the rocket, aluminum would be a better choice than copper due to its lower density and the fact that it also wouldn't melt under the ethanol combustion cycle.

We have determined that using methanol as the working fluid is slightly more efficient than using ethanol by a 1 percent efficiency increase. Compared to industry standard for regenerative cooling engines(15 percent), 12.5 percent is relatively close in consideration of the simplifying assumptions being made in this project.